In [4]:
# Cell 1 — Load feature matrix
import pandas as pd
import plotly.express as px
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
df = pd.read_parquet(PROJECT_ROOT / "data/features/qualifying_features.parquet")
print(df.shape)
df.head()

(2463, 27)


,DeltaToFastest_s,Year,RoundNumber,EventName,Driver,Team,DriverRollingDelta_3,DriverRollingDelta_5,DriverRollingDelta_10,DriverCareerMedianDelta,...,DriverSeasonDeltaTrend,IsStreetCircuit,Altitude_m,TrackTemp,AirTemp,Humidity,WindSpeed,Rainfall,TempDelta,CompoundOrdinal
0,2.204,2022,22,Abu Dhabi Grand Prix,ALB,Williams,1.948667,3.0852,2.4346,2.351,...,NaN,False,3,33.942466,28.763014,69.712329,1.635616,0,5.179452,0
1,0.853,2023,22,Abu Dhabi Grand Prix,ALB,Williams,0.953000,1.2760,2.5188,1.835,...,NaN,False,3,31.846154,26.733333,66.307692,1.510256,0,5.112821,0
2,1.226,2024,24,Abu Dhabi Grand Prix,ALB,Williams,1.675000,1.5390,1.3638,1.592,...,NaN,False,3,29.672840,25.811111,65.790123,1.474074,0,3.861728,0
3,1.209,2025,24,Abu Dhabi Grand Prix,ALB,Williams,3.356667,2.8308,2.0973,1.353,...,NaN,False,3,29.110390,25.096104,76.714286,1.145455,0,4.014286,0
4,2.267,2022,3,Australian Grand Prix,ALB,Williams,9.464000,9.4640,9.4640,9.464,...,2.204,True,20,29.152252,23.237838,57.657658,1.184685,0,5.914414,0


In [5]:
# Cell 2 — Null audit on model input columns
model_cols = [c for c in df.columns if c not in ["Year", "RoundNumber", "EventName", "Driver", "Team"]]
null_summary = df[model_cols].isnull().sum().sort_values(ascending=False)
print(null_summary[null_summary > 0])

DriverCircuitHistoricalDelta    888
TeamCircuitHistoricalDelta      406
DriverSeasonDeltaTrend          132
DriverRollingDelta_5             36
DriverRollingDelta_10            36
DriverCareerMedianDelta          36
DriverRollingDelta_3             36
TeamRollingDelta_3               16
TeamRollingDelta_5               16
TeamCareerMedianDelta            16
dtype: int64


In [6]:
# Cell 3 — Feature distributions
fig = px.histogram(
    df, x="DriverRollingDelta_5", nbins=60,
    title="Distribution of Driver 5-Session Rolling Delta",
    labels={"DriverRollingDelta_5": "Rolling Delta to Fastest (s)"}
)
fig.show()

In [7]:
# Cell 4 — Correlation of all features with target
numeric_df = df.select_dtypes(include="number")
correlations = (
    numeric_df.corr()["DeltaToFastest_s"]
    .drop("DeltaToFastest_s")
    .sort_values()
    .reset_index()
    .rename(columns={"index": "Feature", "DeltaToFastest_s": "Correlation"})
)

fig = px.bar(
    correlations,
    x="Correlation", y="Feature",
    orientation="h",
    color="Correlation",
    color_continuous_scale="RdBu_r",
    title="Feature Correlation with Target (DeltaToFastest_s)",
    height=700
)
fig.add_vline(x=0, line_dash="dash", line_color="white", opacity=0.4)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [8]:
# Cell 5 — Leakage check: confirm no raw lap time columns in feature set
leakage_risk = [c for c in df.columns if "LapTime" in c or "Fastest_s" in c]
print("Potential leakage columns:", leakage_risk)
# Should return only: ['DeltaToFastest_s'] which is the target — that's fine

Potential leakage columns: ['DeltaToFastest_s']


In [9]:
# Cell 6 — First 3 rows per driver to confirm rolling features populated correctly
df[df["Driver"] == df["Driver"].value_counts().index[0]][
    ["Year", "RoundNumber", "EventName",
     "DriverRollingDelta_3", "DriverRollingDelta_5",
     "DriverCircuitHistoricalDelta", "DeltaToFastest_s"]
].head(10)

,Year,RoundNumber,EventName,DriverRollingDelta_3,DriverRollingDelta_5,DriverCircuitHistoricalDelta,DeltaToFastest_s
100,2021,22,Abu Dhabi Grand Prix,1.143667,1.7708,NaN,1.351
101,2022,22,Abu Dhabi Grand Prix,1.065667,0.9538,1.3510,1.272
102,2023,22,Abu Dhabi Grand Prix,0.872333,0.9506,1.3115,0.639
103,2024,24,Abu Dhabi Grand Prix,1.285333,1.1048,1.2720,0.601
104,2025,24,Abu Dhabi Grand Prix,0.951333,1.0042,0.9555,0.654
105,2022,3,Australian Grand Prix,5.963667,4.0286,NaN,0.947
106,2023,3,Australian Grand Prix,0.788333,0.8084,0.9470,0.407
107,2024,3,Australian Grand Prix,0.463333,0.4870,0.6770,0.795
108,2025,1,Australian Grand Prix,1.078333,1.0872,0.7950,1.192
109,2026,1,Australian Grand Prix,1.006000,1.0050,0.8710,3.451
